In [47]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import itertools
import time

start_time = time.time()

train = pd.read_csv("election2000.csv")
test = pd.read_csv("election2000_test.csv")

train.columns = train.columns.str.strip()
test.columns = test.columns.str.strip()

y_train = train["Bush"]
X_train = train.drop(columns=["Bush"])

y_test = test["Bush"]
X_test = test.drop(columns=["Bush"])

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

predictors = list(X_train.columns)

def fit_model(X, y):
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    return model

model1 = fit_model(X_train, y_train)

print(model1.summary())
print("Adjusted R^2:", model1.rsquared_adj)
print("AIC:", model1.aic)
print("BIC:", model1.bic)

print("\nNon-significant predictors (p > 0.05):")
print(model1.pvalues[model1.pvalues > 0.05])

                            OLS Regression Results                            
Dep. Variable:                   Bush   R-squared:                       0.767
Model:                            OLS   Adj. R-squared:                  0.701
Method:                 Least Squares   F-statistic:                     11.67
Date:                Sun, 01 Mar 2026   Prob (F-statistic):           3.28e-09
Time:                        20:41:43   Log-Likelihood:                -154.16
No. Observations:                  51   AIC:                             332.3
Df Residuals:                      39   BIC:                             355.5
Df Model:                          11                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       -700.8994    201.887     -3.472      0.0

In [48]:
best_bic = np.inf
best_features = None
best_model2 = None

for k in range(1, len(predictors)+1):
    for subset in itertools.combinations(predictors, k):
        model = fit_model(X_train[list(subset)], y_train)
        if model.bic < best_bic:
            best_bic = model.bic
            best_features = list(subset)
            best_model2 = model

print("Selected Features:", best_features)
print("Adjusted R^2:", best_model2.rsquared_adj)
print("AIC:", best_model2.aic)
print("BIC:", best_model2.bic)

Selected Features: ['Male', 'Male18', 'NonMet', 'Inc100']
Adjusted R^2: 0.7052940591073735
AIC: 326.05514614349465
BIC: 335.7142743071163


In [49]:
remaining = predictors.copy()
selected = []
current_bic = np.inf

while remaining:
    bic_list = []
    for candidate in remaining:
        features = selected + [candidate]
        model = fit_model(X_train[features], y_train)
        bic_list.append((model.bic, candidate, model))

    bic_list.sort()
    best_new_bic, best_candidate, best_model = bic_list[0]

    if best_new_bic < current_bic:
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        current_bic = best_new_bic
    else:
        break

model3 = best_model
features3 = selected.copy()

print("Selected Features:", features3)
print("Adjusted R^2:", model3.rsquared_adj)
print("AIC:", model3.aic)
print("BIC:", model3.bic)

Selected Features: ['Inc100', 'Male', 'Male18', 'NonMet']
Adjusted R^2: 0.7173542113134326
AIC: 324.80326008364256
BIC: 336.3942138799885


In [50]:
selected = predictors.copy()
current_model = fit_model(X_train[selected], y_train)
current_bic = current_model.bic

while len(selected) > 1:
    bic_list = []
    for candidate in selected:
        temp_features = selected.copy()
        temp_features.remove(candidate)
        model = fit_model(X_train[temp_features], y_train)
        bic_list.append((model.bic, candidate, model))

    bic_list.sort()
    best_new_bic, remove_var, best_model = bic_list[0]

    if best_new_bic < current_bic:
        selected.remove(remove_var)
        current_bic = best_new_bic
        current_model = best_model
    else:
        break

model4 = current_model
features4 = selected.copy()

print("Selected Features:", features4)
print("Adjusted R^2:", model4.rsquared_adj)
print("AIC:", model4.aic)
print("BIC:", model4.bic)

Selected Features: ['Male', 'Male18', 'NonMet', 'Inc100']
Adjusted R^2: 0.7052940591073735
AIC: 326.05514614349465
BIC: 335.7142743071163


In [51]:
kf = KFold(n_splits=5, shuffle=True, random_state=1)

remaining = predictors.copy()
selected = []
best_cv_mse = np.inf

while remaining:
    mse_candidates = []

    for candidate in remaining:
        features = selected + [candidate]
        fold_mse = []

        for train_index, val_index in kf.split(X_train):
            X_tr = X_train.iloc[train_index][features]
            y_tr = y_train.iloc[train_index]
            X_val = X_train.iloc[val_index][features]
            y_val = y_train.iloc[val_index]

            model = fit_model(X_tr, y_tr)
            X_val = sm.add_constant(X_val)
            y_pred = model.predict(X_val)

            fold_mse.append(mean_squared_error(y_val, y_pred))

        mse_candidates.append((np.mean(fold_mse), candidate))

    mse_candidates.sort()
    best_new_mse, best_candidate = mse_candidates[0]

    if best_new_mse < best_cv_mse:
        best_cv_mse = best_new_mse
        selected.append(best_candidate)
        remaining.remove(best_candidate)
    else:
        break

model5 = fit_model(X_train[selected], y_train)
features5 = selected.copy()

print("Selected Features:", features5)
print("Adjusted R^2:", model5.rsquared_adj)
print("AIC:", model5.aic)
print("BIC:", model5.bic)

Selected Features: ['NonMet', 'Inc100', 'Male', 'Male18']
Adjusted R^2: 0.7052940591073782
AIC: 326.05514614349386
BIC: 335.7142743071155


In [52]:
def compute_mse(model):
    exog_names = model.model.exog_names
    
    if 'const' in exog_names:
        features = [x for x in exog_names if x != 'const']
    else:
        features = exog_names.copy()
    
    X_temp = X_test[features].copy()
    X_temp = sm.add_constant(X_temp, has_constant='add')
    
    y_pred = model.predict(X_temp)
    return mean_squared_error(y_test, y_pred)

In [53]:
mse1 = compute_mse(model1)
mse2 = compute_mse(best_model2)
mse3 = compute_mse(model3)
mse4 = compute_mse(model4)
mse5 = compute_mse(model5)
print("Model 1 MSE:", mse1)
print("Model 2 MSE:", mse2)
print("Model 3 MSE:", mse3)
print("Model 4 MSE:", mse4)
print("Model 5 MSE:", mse5)

Model 1 MSE: 59.10540043147962
Model 2 MSE: 56.427484803578274
Model 3 MSE: 61.235283822693795
Model 4 MSE: 56.427484803578274
Model 5 MSE: 56.427484803567594


In [54]:
comparison = pd.DataFrame({
    "Model": ["Full", "BestSubset", "Forward", "Backward", "ForwardCV"],
    "Adj_R2": [model1.rsquared_adj,
               best_model2.rsquared_adj,
               model3.rsquared_adj,
               model4.rsquared_adj,
               model5.rsquared_adj],
    "AIC": [model1.aic,
            best_model2.aic,
            model3.aic,
            model4.aic,
            model5.aic],
    "BIC": [model1.bic,
            best_model2.bic,
            model3.bic,
            model4.bic,
            model5.bic],
    "Test_MSE": [mse1, mse2, mse3, mse4, mse5]
})

print(comparison)

        Model    Adj_R2         AIC         BIC   Test_MSE
0        Full  0.701280  332.326105  355.508013  59.105400
1  BestSubset  0.705294  326.055146  335.714274  56.427485
2     Forward  0.717354  324.803260  336.394214  61.235284
3    Backward  0.705294  326.055146  335.714274  56.427485
4   ForwardCV  0.705294  326.055146  335.714274  56.427485


In [57]:
def print_coefficients(model, model_name):
    print(f"\n{model_name} Coefficients")
    coef_table = model.params
    for var, coef in coef_table.items():
        print(f"{var}: {coef:.4f}")

print_coefficients(model1, "Model 1")
print_coefficients(best_model2, "Model 2")
print_coefficients(model3, "Model 3")
print_coefficients(model4, "Model 4")
print_coefficients(model5, "Model 5")


Model 1 Coefficients
const: -700.8994
UnE: -1.7352
Pop: -0.0000
Male: 56.9342
Male18: -41.9787
Pop65: -0.9846
NonMet: 0.1816
Pov: 0.4061
NuHouse: 0.0000
Inc50: -0.3492
Inc75: 1.3514
Inc100: -3.1018

Model 2 Coefficients
const: -802.2008
Male: 61.9478
Male18: -45.2570
NonMet: 0.1431
Inc100: -1.7563

Model 3 Coefficients
const: -717.4201
Inc100: -2.0361
Male: 59.5678
Male18: -44.3473
NonMet: 0.1486
Pop65: -0.8928

Model 4 Coefficients
const: -802.2008
Male: 61.9478
Male18: -45.2570
NonMet: 0.1431
Inc100: -1.7563

Model 5 Coefficients
const: -802.2008
NonMet: 0.1431
Inc100: -1.7563
Male: 61.9478
Male18: -45.2570
